In [2]:
from huggingface_hub import snapshot_download
import os

# Path where we will store model temporarily
local_dir = "/kaggle/working/bge-m3"

# Download model from HuggingFace
snapshot_download(
    repo_id="BAAI/bge-m3",
    local_dir=local_dir,
    local_dir_use_symlinks=False
)

print("Model downloaded to:", local_dir)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Model downloaded to: /kaggle/working/bge-m3


In [3]:
from sentence_transformers import SentenceTransformer

# Replace with your dataset path name
model_path = "/kaggle/working/bge-m3"

model = SentenceTransformer(model_path)

# Test
emb = model.encode("Hello Dhairya, trading AI is coming 🔥")
print(emb[:5])

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[-0.0365409  -0.01357438 -0.03621339  0.03791505 -0.02616842]


In [4]:
import json
import os

# Define paths
input_path = '/kaggle/input/datasets/dhairya4252/1-my-dataset/1-reranked.json'
output_path = '/kaggle/working/1-reranked_cleared.json'

try:
    # 1. Load the original data (optional, just to verify it exists)
    with open(input_path, 'r') as f:
        data = json.load(f)
    print(f"Original file loaded. It contained {len(data)} items.")

    # 2. Define the 'cleared' state (usually an empty list or dict)
    empty_data = [] 

    # 3. Write the empty data to the writable /kaggle/working/ directory
    with open(output_path, 'w') as f:
        json.dump(empty_data, f)
    
    print(f"Success! A cleared version of the file has been saved to: {output_path}")

except FileNotFoundError:
    print("Error: The input file path was not found. Please check the dataset connection.")
except Exception as e:
    print(f"An error occurred: {e}")

An error occurred: Expecting value: line 1 column 1 (char 0)


# main loop

In [5]:
pip install FlagEmbedding

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 104.4 kB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 21.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.6 MB/s eta 0:00:00
  Created wheel for FlagEmbedding: filename=FlagEmbedding-1.3.5-py3-none-any.whl size=233746 sha256=b23b44d03b621a313e17504c1691674d52976858fad02e9323a8bdb97ffdabcb
  Stored in directory: /root/.cache/pip/wheels/b2/1f/f6/78f862bb80cb959cc9960b7c4e2d1f702b1bc0e79d19b5f124
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18919 sha256=382e49eda2ede7669bdc2d7b91ed286011a2cbe613d82b5619b48776b8a98356
  Stored in directory

In [6]:
pip install -q sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import json
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm


# ============================================================================
# 1. LOAD MODEL
# ============================================================================
print("Loading BGE-M3 model...")
model = SentenceTransformer('/kaggle/working/bge-m3')

if torch.cuda.is_available():
    model = model.to('cuda')

print("Model loaded successfully!\n")

# ============================================================================
# 2. LOAD JSON FILES
# ============================================================================
print("Loading JSON files...")
with open('/kaggle/input/datasets/dhairya4252/1-my-dataset/1-concatenated-claims.json', 'r') as f:
    decomposed_data = json.load(f)

with open('/kaggle/input/datasets/dhairya4252/1-my-dataset/1-top100-BM25.json', 'r') as f:
    bm25_data = json.load(f)

print(f"Loaded {len(decomposed_data)} entries from decomposed file")
print(f"Loaded {len(bm25_data)} entries from BM25 file\n")

# ============================================================================
# 3. CREATE CLAIM LOOKUP FOR FAST MATCHING
# ============================================================================
print("Creating claim lookup dictionary...")
bm25_lookup = {entry['claim']: entry for entry in bm25_data}
print(f"Created lookup with {len(bm25_lookup)} unique claims\n")

# ============================================================================
# 4. PROCESS EACH MATCHED PAIR
# ============================================================================
print("Processing matched claims...")
results = []
batch_size = 16  # Adjust based on your GPU memory

if torch.cuda.is_available():
    device = torch.device('cuda')
    batch_size = 16  # GPU - can handle larger batches
    print(f"🚀 Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
    batch_size = 16  # CPU - smaller batches
    print("⚠️ Using CPU - this will be slower")


for decomposed_entry in tqdm(decomposed_data, desc="Processing claims"):
    claim = decomposed_entry.get('claim', '')
    
    # Match claim with BM25 data
    if claim not in bm25_lookup:
        print(f"Warning: Claim not found in BM25 data: {claim[:50]}...")
        continue
    
    bm25_entry = bm25_lookup[claim]
    
    # Extract data
    concatenated_claims = decomposed_entry.get('concatenated_claims', [])
    docs = bm25_entry.get('docs', [])
    original_id = decomposed_entry.get('original_id', None)
    
    # Validate data
    if len(concatenated_claims) != 5:
        print(f"Warning: Expected 5 concatenated_claims, got {len(concatenated_claims)}")
        continue
    
    if len(docs) != 100:
        print(f"Warning: Expected 100 docs, got {len(docs)}")
        continue
    
    # ========================================================================
    # 5. PREPARE TEXTS WITH INSTRUCTIONS (BGE-M3 format)
    # ========================================================================
    query_texts = [f"Represent this claim for retrieval: {q}" for q in concatenated_claims]
    doc_texts = [f"Represent this document for retrieval: {d}" for d in docs]
    
    # ========================================================================
    # 6. ENCODE WITH BATCHING AND NORMALIZATION
    # ========================================================================
    # Encode queries (5 claims)
    query_embeddings = model.encode(
        query_texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    
    # Encode documents (100 docs)
    doc_embeddings = model.encode(
        doc_texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    
    # ========================================================================
    # 7. COMPUTE COSINE SIMILARITY (dot product for normalized vectors)
    # ========================================================================
    # Shape: (5 queries, 100 docs)
    similarity_matrix = np.dot(query_embeddings, doc_embeddings.T)
    
    # ========================================================================
    # 8. GET TOP 1 DOC FOR EACH QUERY
    # ========================================================================
    top_docs = []
    top_scores = []
    
    for i in range(len(concatenated_claims)):
        # Get similarity scores for this query
        scores = similarity_matrix[i]
        
        # Get index of top document
        top_idx = np.argmax(scores)
        
        # Store result
        top_docs.append(docs[top_idx])
        top_scores.append(float(scores[top_idx]))
    
    # ========================================================================
    # 9. CREATE OUTPUT ENTRY
    # ========================================================================
    result_entry = {
        'claim': claim,
        'original_id': original_id,
        'docs': top_docs,  # List of 5 best docs (one per concatenated_claim)
        'scores': top_scores  # List of 5 scores
    }
    
    results.append(result_entry)
    torch.cuda.empty_cache()

print(f"\nProcessed {len(results)} matched claims successfully!\n")

# ============================================================================
# 10. SAVE RESULTS
# ============================================================================
output_path = '/kaggle/working/1-reranked.json'
print(f"Saving results to {output_path}...")

with open(output_path, 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"✅ Results saved successfully!")
print(f"Total entries: {len(results)}")

# ============================================================================
# 11. SAMPLE OUTPUT
# ============================================================================
if results:
    print("\n" + "="*80)
    print("SAMPLE OUTPUT:")
    print("="*80)
    sample = results[0]
    print(f"Claim: {sample['claim'][:100]}...")
    print(f"Original ID: {sample['original_id']}")
    print(f"\nTop docs (one per concatenated claim):")
    for i, (doc, score) in enumerate(zip(sample['docs'], sample['scores']), 1):
        print(f"\n  [{i}] Score: {score:.4f}")
        print(f"      Doc: {doc[:100]}...")

Loading BGE-M3 model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model loaded successfully!

Loading JSON files...
Loaded 9430 entries from decomposed file
Loaded 9935 entries from BM25 file

Creating claim lookup dictionary...
Created lookup with 9929 unique claims

Processing matched claims...
🚀 Using GPU: Tesla T4



Processing claims:   0%|          | 0/9430 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]


Processing claims:  20%|█▉        | 1846/9430 [1:13:40<5:46:26,  2.74s/it]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]